# Notebook 01 — Initial Exploration

**Project:** How to Successfully Transition into Tech: A Data-Driven Analysis  
**Author:** Víctor Aguilar — Data Analytics Instructor  
**Dataset:** Stack Overflow Developer Survey 2024 (65,437 respondents from 185 countries)

---

## Purpose of this notebook

**Technique taught:** Initial Exploratory Data Analysis (EDA)  
**Goal:** Understand the dataset's shape, quality, and what questions we can actually answer with it.

## What students will learn

1. How to load a large CSV efficiently with pandas
2. The first 5 questions any analyst asks a new dataset
3. How to identify columns relevant to a research question
4. How to spot data quality issues *before* analyzing

## Research question

> *Who transitions into tech successfully, and what do they have in common?*

Before answering, we need to know what the data can tell us.

## 1. Setup and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
sns.set_theme(style='whitegrid')

# --- For Google Colab users ---
# Uncomment these lines if running on Colab:
# from google.colab import files
# uploaded = files.upload()  # Then upload survey_results_public.csv

# --- For local execution ---
DATA_PATH = '../data/survey_results_public.csv'  # Adjust if needed

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Rows:    {df.shape[0]:>7,}')
print(f'Columns: {df.shape[1]:>7,}')

## 2. The 5 questions every analyst asks first

When you receive a new dataset, always answer these *in order*:

1. **What does each row represent?**
2. **What columns exist and what do they mean?**
3. **How much missing data is there?**
4. **What types of variables do we have?** (numeric / categorical / multi-valued)
5. **Are there obvious data quality issues?**

### Q1 — What does each row represent?

Each row is **one developer's survey response**. The `ResponseId` column is the unique identifier.

In [ ]:
df[['ResponseId', 'MainBranch', 'Country', 'Age']].head()

### Q2 — What columns exist?

114 columns is a lot. Let's group them by topic to understand the survey's structure.

In [ ]:
# All column names
all_cols = df.columns.tolist()
print(f'Total columns: {len(all_cols)}')
print()
for i, col in enumerate(all_cols, 1):
    print(f'{i:>3}. {col}')

**Observation:** Many columns come in triplets like `LanguageHaveWorkedWith`, `LanguageWantToWorkWith`, `LanguageAdmired`. These are **multi-valued columns** (semicolon-separated). We'll handle them in Notebook 02.

### Columns relevant to OUR research question

For the career-transition question, these are the most useful:

In [ ]:
# Columns we care about for career transition analysis
key_cols = {
    'Demographics':  ['Age', 'Country', 'MainBranch'],
    'Education':     ['EdLevel', 'LearnCode', 'LearnCodeOnline'],
    'Experience':    ['YearsCode', 'YearsCodePro', 'DevType'],
    'Compensation':  ['CompTotal', 'Currency', 'ConvertedCompYearly'],
    'Work setup':    ['RemoteWork', 'OrgSize', 'Employment'],
    'Skills':        ['LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith',
                      'PlatformHaveWorkedWith']
}

for category, cols in key_cols.items():
    print(f'\n--- {category} ---')
    for c in cols:
        exists = '✓' if c in df.columns else '✗ MISSING'
        print(f'  {exists}  {c}')

### Q3 — How much missing data?

**Pedagogical note:** Missing data is rarely random. Understanding *why* it's missing is often more important than the missingness itself.

In [ ]:
# Missing data for our key columns only
key_flat = [c for cols in key_cols.values() for c in cols if c in df.columns]

missing = pd.DataFrame({
    'missing_count':   df[key_flat].isna().sum(),
    'missing_percent': (df[key_flat].isna().mean() * 100).round(1)
}).sort_values('missing_percent', ascending=False)

missing

In [ ]:
# Visualize missingness
fig, ax = plt.subplots(figsize=(10, 6))
missing['missing_percent'].sort_values().plot(
    kind='barh', ax=ax, color='#c0392b'
)
ax.set_xlabel('Missing values (%)')
ax.set_title('Missing data in key columns')
ax.axvline(50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/01_missing_data.png', dpi=120, bbox_inches='tight')
plt.show()

**Insight for students:**

- `CompTotal` and `Currency` have high missingness — many respondents skip salary questions.
- `YearsCodePro` is missing for everyone who isn't a professional yet (learners, hobbyists).
- This is **structural missingness**, not random. Filtering, not imputation, is the right approach.

### Q4 — What types of variables?

In [ ]:
df[key_flat].dtypes.value_counts()

In [ ]:
# Most categorical columns are strings. Show example values:
for col in ['Age', 'EdLevel', 'RemoteWork', 'MainBranch']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False).head(8))

### Q5 — Data quality issues to flag

In [ ]:
# YearsCode and YearsCodePro contain text values like 'Less than 1 year', 'More than 50 years'
print('YearsCode unique values (sample):')
print(df['YearsCode'].unique()[:15])
print()
print('YearsCodePro unique values (sample):')
print(df['YearsCodePro'].unique()[:15])

**Issue identified:** `YearsCode` and `YearsCodePro` are stored as strings due to text values. We'll convert them to numeric in Notebook 02.

## 3. First glimpse at our research question

Before any cleaning, let's see what the raw data hints at.

In [ ]:
# Who is in the dataset?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['MainBranch'].value_counts().plot(
    kind='barh', ax=axes[0], color='#2980b9'
)
axes[0].set_title('Respondent profile')
axes[0].set_xlabel('Count')

df['Country'].value_counts().head(15).plot(
    kind='barh', ax=axes[1], color='#16a085'
)
axes[1].set_title('Top 15 countries')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/figures/01_who_is_in_dataset.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Colombia spotlight (since this matters to our students in Cali)
colombia_count = (df['Country'] == 'Colombia').sum()
total = len(df)
print(f'Colombian respondents: {colombia_count:,} ({colombia_count/total*100:.2f}% of dataset)')
print(f'Enough for a sub-analysis? {"Yes" if colombia_count >= 100 else "Borderline"}.')

## 4. Summary and next steps

### What we learned about the dataset

| Aspect | Finding |
|---|---|
| Size | 65,437 rows × 114 columns |
| Granularity | One row per developer |
| Coverage | 185 countries; top 10 account for ~60% |
| Colombia subsample | 217 respondents — viable for case study |
| Multi-valued columns | Many (semicolon-separated); need expansion |
| Numeric-as-string | `YearsCode`, `YearsCodePro` need conversion |
| Missing data | Mostly structural (e.g., salary, pro experience) |

### What's next

**Notebook 02 — Data Cleaning** will:
- Convert `YearsCode` / `YearsCodePro` to numeric
- Expand multi-valued columns into binary indicators
- Filter to professional developers for the career-transition analysis
- Save a cleaned dataset for downstream analysis